# Arabic Dialect → MSA | Colab GPU starter

Run cells in order after selecting **Runtime → Change runtime type → T4 GPU**. This is a reproducible baseline experiment, **not yet the new dictionary-data experiment**. It runs on the existing training data and preserves the original `main` branch. Colab sessions can disconnect.

In [ ]:
!nvidia-smi
!pip -q install transformers==4.46.3 sentencepiece accelerate sacrebleu pandas tqdm scikit-learn protobuf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BACKUP = Path('/content/drive/MyDrive/arabic-dialect-to-msa-experiments')
BACKUP.mkdir(parents=True, exist_ok=True)
print('Backup:', BACKUP)

In [ ]:
import os, subprocess
REPO='/content/arabic-dialect-to-msa'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--branch','dialect-data-improvement','--single-branch','https://github.com/saraabduallh-1/arabic-dialect-to-msa.git',REPO],check=True)
os.chdir(REPO)
print(subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True))

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU unavailable: select T4 GPU in Runtime settings.'
print(torch.cuda.get_device_name(0))
!python src/finetune.py --model UBC-NLP/AraT5v2-base-1024 --epochs 1 --bs 2 --optim adafactor --limit 16 --tag colab-smoke

## Full run

Only execute after the smoke test passes. The existing script selects checkpoints on dev, but evaluates on test at the end; for rigorous final comparison use a fresh held-out test later. This training script does **not** resume optimizer state automatically if Colab disconnects. Saved artifacts are copied to Drive only when the command completes.

In [ ]:
import subprocess, shutil, datetime, json
cmd=['python','src/finetune.py','--model','UBC-NLP/AraT5v2-base-1024','--epochs','5','--bs','4','--optim','adafactor','--tag','+colab-baseline']
log=BACKUP/'colab_baseline.log'
with open(log,'a',encoding='utf-8') as f:
    f.write('\nRUN '+datetime.datetime.now().isoformat()+' '+str(cmd)+'\n');f.flush()
    result=subprocess.run(cmd,cwd=REPO,stdout=f,stderr=subprocess.STDOUT)
print('Exit code:',result.returncode,'Log:',log)
for rel in ['results/scores.jsonl','results/scores_dev.jsonl','results/preds/AraT5v2-base-1024+colab-baseline__finetune.jsonl']:
    p=Path(REPO)/rel
    if p.exists():
        dest=BACKUP/rel;dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dest)
ck=Path(REPO)/'results/checkpoints/AraT5v2-base-1024+colab-baseline'
if ck.exists():
    shutil.copytree(ck,BACKUP/'checkpoint_colab_baseline',dirs_exist_ok=True)
if result.returncode: raise RuntimeError('Training failed. Check log on Google Drive.')
print('Saved baseline outputs to Google Drive.')